# Phase 6: RAG & Vector Databases
## Day 26: EmbeddingsDeepDive

Date: 2026-04-24

### Learning objectives
- Understand what embeddings are.
- Create simple local embeddings.
- Calculate cosine similarity.
- Use OpenAI embedding API patterns safely.
- Try sentence-transformers with a local fallback.
- Build a tiny semantic search system.

In [ ]:
import os
import json
import math
import hashlib
import textwrap
from pprint import pprint

import numpy as np
import pandas as pd

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    SKLEARN_AVAILABLE = True
except Exception:
    TfidfVectorizer = None
    SKLEARN_AVAILABLE = False

try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_TRANSFORMERS_AVAILABLE = True
except Exception:
    SentenceTransformer = None
    SENTENCE_TRANSFORMERS_AVAILABLE = False

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except Exception:
    OpenAI = None
    OPENAI_AVAILABLE = False

def show(title, content):
    print("\n" + "=" * 82)
    print(title)
    print("=" * 82)
    print(textwrap.dedent(str(content)).strip())

np.set_printoptions(precision=4, suppress=True)

print("Setup complete.")
print("scikit-learn available:", SKLEARN_AVAILABLE)
print("sentence-transformers available:", SENTENCE_TRANSFORMERS_AVAILABLE)
print("openai package available:", OPENAI_AVAILABLE)
print("OPENAI_API_KEY exists:", bool(os.getenv("OPENAI_API_KEY")))

In [ ]:
documents = [
    {
        "doc_id": "C001",
        "category": "campaign",
        "text": "Spring Coffee Push on Instagram had strong clicks and good conversions from a limited-time discount."
    },
    {
        "doc_id": "C002",
        "category": "campaign",
        "text": "Bank App Onboarding email campaign had low conversion and needs a better subject line."
    },
    {
        "doc_id": "C003",
        "category": "campaign",
        "text": "Yoga Studio Trial on TikTok performed well with beginner-friendly short videos."
    },
    {
        "doc_id": "O001",
        "category": "ocr",
        "text": "OCR pipeline extracts invoice ID, date, total amount, and campaign fields from scanned documents."
    },
    {
        "doc_id": "O002",
        "category": "ocr",
        "text": "OpenCV preprocessing improves OCR by using grayscale, thresholding, denoising, and deskewing."
    },
    {
        "doc_id": "R001",
        "category": "rag",
        "text": "RAG systems retrieve relevant chunks from a vector database before generating an answer."
    },
    {
        "doc_id": "R002",
        "category": "rag",
        "text": "Chunking strategy affects retrieval quality, context length, and answer accuracy."
    },
    {
        "doc_id": "F001",
        "category": "food",
        "text": "A sourdough bagel with cream cheese and coffee is a relaxing weekend breakfast."
    }
]

df = pd.DataFrame(documents)
df

## 1. What is an embedding?

An embedding is a list of numbers that represents meaning.

Similar texts should have vectors that point in similar directions. That makes search and retrieval possible.

In [ ]:
tiny_embedding_example = {
    "text": "coffee campaign",
    "embedding": [0.21, 0.84, -0.12, 0.34]
}

pprint(tiny_embedding_example)

print("Vector length:", len(tiny_embedding_example["embedding"]))
print("Vector type:", type(tiny_embedding_example["embedding"]))

In [ ]:
def vector_length(vector):
    vector = np.array(vector, dtype=float)
    return np.sqrt(np.sum(vector ** 2))

v = np.array([3, 4])
print("Vector:", v)
print("Length:", vector_length(v))

## 2. Cosine similarity

Cosine similarity measures whether two vectors point in the same direction.

It is common for embedding search because it focuses on semantic direction, not raw vector size.

In [ ]:
def cosine_similarity(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)

    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return 0.0

    return float(np.dot(a, b) / denominator)

vectors = {
    "coffee": np.array([1.0, 0.9, 0.1]),
    "latte": np.array([0.9, 1.0, 0.1]),
    "invoice": np.array([0.1, 0.1, 1.0]),
}

print("coffee vs latte:", round(cosine_similarity(vectors["coffee"], vectors["latte"]), 3))
print("coffee vs invoice:", round(cosine_similarity(vectors["coffee"], vectors["invoice"]), 3))

In [ ]:
pairs = []

for left_name, left_vector in vectors.items():
    for right_name, right_vector in vectors.items():
        pairs.append({
            "left": left_name,
            "right": right_name,
            "cosine_similarity": round(cosine_similarity(left_vector, right_vector), 3)
        })

pd.DataFrame(pairs)

## 3. Local embeddings with TF-IDF

TF-IDF is not a neural embedding model.

But it is useful for learning because it turns text into vectors and lets us practice similarity search locally.

In [ ]:
if SKLEARN_AVAILABLE:
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(df["text"])
    local_embeddings = tfidf_matrix.toarray()
    feature_names = vectorizer.get_feature_names_out()
else:
    local_embeddings = None
    feature_names = []

print("Embedding matrix shape:", None if local_embeddings is None else local_embeddings.shape)
print("First 15 features:", list(feature_names[:15]))

In [ ]:
def simple_hash_embedding(text, dim=32):
    # Fallback embedding when scikit-learn is not available.
    vector = np.zeros(dim, dtype=float)
    words = text.lower().split()

    for word in words:
        digest = hashlib.md5(word.encode("utf-8")).hexdigest()
        index = int(digest[:8], 16) % dim
        sign = 1 if int(digest[8:16], 16) % 2 == 0 else -1
        vector[index] += sign

    norm = np.linalg.norm(vector)
    return vector if norm == 0 else vector / norm

if local_embeddings is None:
    local_embeddings = np.vstack([simple_hash_embedding(text) for text in df["text"]])
    print("Using hash fallback embeddings.")

print("Local embeddings shape:", local_embeddings.shape)
print("First vector preview:", local_embeddings[0][:10])

## 4. Semantic search with cosine similarity

Semantic search embeds the query and compares it with document embeddings.

The closest vectors become the top search results.

In [ ]:
def embed_query_local(query):
    if SKLEARN_AVAILABLE:
        return vectorizer.transform([query]).toarray()[0]
    return simple_hash_embedding(query, dim=local_embeddings.shape[1])

def search_documents(query, top_k=3):
    query_embedding = embed_query_local(query)

    scores = [
        cosine_similarity(query_embedding, doc_embedding)
        for doc_embedding in local_embeddings
    ]

    results = df.copy()
    results["score"] = scores
    return results.sort_values("score", ascending=False).head(top_k)

search_documents("Which document talks about invoice OCR extraction?", top_k=3)

In [ ]:
queries = [
    "campaign had low conversion",
    "OCR preprocessing with thresholding",
    "vector database retrieval",
    "coffee breakfast"
]

for query in queries:
    print("\nQuery:", query)
    display(search_documents(query, top_k=2)[["doc_id", "category", "score", "text"]])

## 5. OpenAI embeddings API pattern

OpenAI embeddings are produced by an API model.

The code below is safe. It only calls the API when the OpenAI package and `OPENAI_API_KEY` are available.

In [ ]:
openai_embedding_notes = '''
Current common model choices:
- text-embedding-3-small
- text-embedding-3-large

Basic Python shape:
from openai import OpenAI
client = OpenAI()

response = client.embeddings.create(
    model="text-embedding-3-small",
    input="Your text string goes here"
)

embedding = response.data[0].embedding
'''

print(openai_embedding_notes)

In [ ]:
def get_openai_embedding_safe(text, model="text-embedding-3-small"):
    if not OPENAI_AVAILABLE:
        print("openai package is not installed. Using local fallback embedding.")
        return simple_hash_embedding(text, dim=64)

    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY is missing. Using local fallback embedding.")
        return simple_hash_embedding(text, dim=64)

    client = OpenAI()
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return np.array(response.data[0].embedding, dtype=float)

example_embedding = get_openai_embedding_safe("A campaign about coffee and Instagram.")
print("Embedding length:", len(example_embedding))
print("Preview:", example_embedding[:8])

In [ ]:
def get_openai_embeddings_batch_safe(texts, model="text-embedding-3-small"):
    if not OPENAI_AVAILABLE or not os.getenv("OPENAI_API_KEY"):
        print("Using local fallback batch embeddings.")
        return np.vstack([simple_hash_embedding(text, dim=64) for text in texts])

    client = OpenAI()
    response = client.embeddings.create(
        model=model,
        input=texts
    )

    return np.array([item.embedding for item in response.data], dtype=float)

batch_embeddings = get_openai_embeddings_batch_safe(df["text"].tolist())
print("Batch embedding shape:", batch_embeddings.shape)

## 6. Sentence-transformers pattern

`sentence-transformers` runs embedding models locally.

It is useful when you want local embeddings without sending text to an external API.

In [ ]:
sentence_transformer_notes = '''
Install:
pip install sentence-transformers

Common starter model:
sentence-transformers/all-MiniLM-L6-v2

Basic usage:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(texts, normalize_embeddings=True)
'''

print(sentence_transformer_notes)

In [ ]:
def get_sentence_transformer_embeddings_safe(texts, model_name="sentence-transformers/all-MiniLM-L6-v2"):
    if not SENTENCE_TRANSFORMERS_AVAILABLE:
        print("sentence-transformers is not installed. Using local fallback embeddings.")
        return np.vstack([simple_hash_embedding(text, dim=64) for text in texts])

    try:
        model = SentenceTransformer(model_name)
        return model.encode(texts, normalize_embeddings=True)
    except Exception as error:
        print("Could not load sentence-transformers model. Using fallback embeddings.")
        print("Error:", error)
        return np.vstack([simple_hash_embedding(text, dim=64) for text in texts])

st_embeddings = get_sentence_transformer_embeddings_safe(df["text"].tolist())
print("Sentence-transformer style embedding shape:", st_embeddings.shape)

In [ ]:
def search_with_embeddings(query, doc_embeddings, embed_function, top_k=3):
    query_embedding = embed_function(query)

    scores = [
        cosine_similarity(query_embedding, doc_embedding)
        for doc_embedding in doc_embeddings
    ]

    results = df.copy()
    results["score"] = scores
    return results.sort_values("score", ascending=False).head(top_k)

def embed_query_hash64(query):
    return simple_hash_embedding(query, dim=st_embeddings.shape[1])

search_with_embeddings(
    query="Find something about retrieval and vector databases",
    doc_embeddings=st_embeddings,
    embed_function=embed_query_hash64,
    top_k=3
)

## 7. Normalization and dot product

Many embedding systems normalize vectors to length 1.

When vectors are normalized, cosine similarity and dot product give the same ranking.

In [ ]:
def normalize_l2(vector):
    vector = np.array(vector, dtype=float)
    norm = np.linalg.norm(vector)
    if norm == 0:
        return vector
    return vector / norm

a = np.array([3, 4])
b = np.array([6, 8])

a_norm = normalize_l2(a)
b_norm = normalize_l2(b)

print("Original a length:", np.linalg.norm(a))
print("Normalized a length:", np.linalg.norm(a_norm))
print("Cosine similarity:", cosine_similarity(a, b))
print("Dot product after normalization:", float(np.dot(a_norm, b_norm)))

In [ ]:
def normalize_matrix(matrix):
    matrix = np.array(matrix, dtype=float)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return np.where(norms == 0, matrix, matrix / norms)

normalized_embeddings = normalize_matrix(local_embeddings)

query = "OCR invoice extraction"
query_embedding = normalize_l2(embed_query_local(query))

dot_scores = normalized_embeddings @ query_embedding
cos_scores = np.array([cosine_similarity(query_embedding, row) for row in normalized_embeddings])

print("Dot scores:", np.round(dot_scores[:5], 4))
print("Cos scores:", np.round(cos_scores[:5], 4))
print("Same ranking:", np.argsort(-dot_scores).tolist() == np.argsort(-cos_scores).tolist())

## 8. Dimensions and storage

An embedding dimension is the number of values in the vector.

Higher dimensions can capture more information, but they also cost more memory and search time.

In [ ]:
def estimate_embedding_storage(num_vectors, dimensions, bytes_per_value=4):
    total_bytes = num_vectors * dimensions * bytes_per_value
    return {
        "num_vectors": num_vectors,
        "dimensions": dimensions,
        "bytes": total_bytes,
        "mb": round(total_bytes / (1024 ** 2), 2),
        "gb": round(total_bytes / (1024 ** 3), 4)
    }

storage_examples = pd.DataFrame([
    estimate_embedding_storage(1_000, 384),
    estimate_embedding_storage(100_000, 384),
    estimate_embedding_storage(100_000, 1536),
    estimate_embedding_storage(1_000_000, 1536),
    estimate_embedding_storage(1_000_000, 3072),
])

storage_examples

In [ ]:
def explain_dimension_choice(dimensions):
    if dimensions <= 384:
        return "Small and fast. Good for local demos and lightweight search."
    if dimensions <= 1536:
        return "Balanced. Common for production semantic search."
    return "Large. May improve quality but needs more memory and compute."

for dim in [384, 768, 1536, 3072]:
    print(dim, "=>", explain_dimension_choice(dim))

## 9. Tiny recommendation system

Embeddings can recommend similar items.

Here we find documents similar to a selected document.

In [ ]:
def recommend_similar_documents(source_doc_id, embeddings, top_k=3):
    source_index = df.index[df["doc_id"] == source_doc_id][0]
    source_embedding = embeddings[source_index]

    scores = [
        cosine_similarity(source_embedding, doc_embedding)
        for doc_embedding in embeddings
    ]

    results = df.copy()
    results["score"] = scores
    results = results[results["doc_id"] != source_doc_id]
    return results.sort_values("score", ascending=False).head(top_k)

recommend_similar_documents("R001", local_embeddings, top_k=3)

## 10. Evaluate retrieval with simple expectations

In real RAG systems, you need retrieval evaluation.

A simple start is checking whether expected documents appear in the top results.

In [ ]:
test_cases = [
    {
        "query": "invoice OCR fields",
        "expected_doc_ids": {"O001", "O002"}
    },
    {
        "query": "email campaign conversion problem",
        "expected_doc_ids": {"C002"}
    },
    {
        "query": "retrieval from vector database",
        "expected_doc_ids": {"R001", "R002"}
    },
]

def evaluate_retrieval(test_cases, top_k=3):
    rows = []

    for case in test_cases:
        results = search_documents(case["query"], top_k=top_k)
        retrieved = set(results["doc_id"])
        expected = case["expected_doc_ids"]
        hit = len(retrieved & expected) > 0

        rows.append({
            "query": case["query"],
            "expected": sorted(expected),
            "retrieved": results["doc_id"].tolist(),
            "hit": hit
        })

    return pd.DataFrame(rows)

evaluation_df = evaluate_retrieval(test_cases, top_k=3)
evaluation_df

In [ ]:
hit_rate = evaluation_df["hit"].mean()
print("Retrieval hit rate:", round(hit_rate, 3))

## Tricky bits

Embeddings feel simple, but production search has many traps.

The biggest problems are bad chunks, mixed embedding models, missing normalization, and no retrieval evaluation.

In [ ]:
tricky_bits = pd.DataFrame([
    {
        "problem": "Using different models for documents and queries",
        "why_it_hurts": "Vectors may not live in the same semantic space",
        "fix": "Use the same embedding model for indexing and querying"
    },
    {
        "problem": "No retrieval evaluation",
        "why_it_hurts": "You cannot tell if search quality is improving",
        "fix": "Create test queries with expected results"
    },
    {
        "problem": "Bad chunking",
        "why_it_hurts": "Relevant information may be split or buried",
        "fix": "Tune chunk size and overlap"
    },
    {
        "problem": "Ignoring cost and dimensions",
        "why_it_hurts": "Large vectors increase storage and search cost",
        "fix": "Estimate storage before indexing"
    },
    {
        "problem": "Searching raw long documents",
        "why_it_hurts": "The vector may represent too many ideas at once",
        "fix": "Chunk documents before embedding"
    },
])

tricky_bits

In [ ]:
def diagnose_search_problem(symptom):
    symptom = symptom.lower()
    if "wrong results" in symptom or "irrelevant" in symptom:
        return "Check chunking, embedding model, and retrieval evaluation."
    if "slow" in symptom:
        return "Check vector dimension, index type, and number of vectors."
    if "expensive" in symptom:
        return "Reduce dimensions, batch requests, and avoid re-embedding unchanged text."
    if "missing" in symptom:
        return "Check whether the source text was indexed and chunked correctly."
    return "Inspect query, chunks, model, and similarity scores."

for symptom in [
    "Search returns irrelevant results",
    "Search is slow",
    "Embedding is expensive",
    "Important answers are missing"
]:
    print(symptom, "=>", diagnose_search_problem(symptom))

## Trick questions

1. Are embeddings human-readable?

<details>
<summary>Answer</summary>

No. They are numeric vectors. We interpret them through similarity, clustering, search, or visualization.

</details>

2. Should you embed documents and queries with different models?

<details>
<summary>Answer</summary>

Usually no. Use the same model so the vectors live in the same semantic space.

</details>

3. What does cosine similarity measure?

<details>
<summary>Answer</summary>

It measures how similar two vector directions are.

</details>

4. Why normalize embeddings?

<details>
<summary>Answer</summary>

Normalization makes vector length equal to 1. Then dot product can behave like cosine similarity.

</details>

5. Why evaluate retrieval?

<details>
<summary>Answer</summary>

Because retrieval quality controls the quality of the RAG answer. Bad retrieval leads to bad context.

</details>

## Exercises

Fill in each `___`. Run the cell to check your answer.

In [ ]:
# Exercise 1
# Create a numpy vector from a list.

vector = ___

assert isinstance(vector, np.ndarray)
assert vector.shape == (3,)
print("Exercise 1 passed.")

In [ ]:
# Exercise 2
# Calculate cosine similarity between two vectors.

a = np.array([1, 0, 0])
b = np.array([1, 1, 0])

score = ___

assert 0.70 < score < 0.72
print("Exercise 2 passed.")

In [ ]:
# Exercise 3
# Embed a query using the local helper.

query_embedding = ___

assert isinstance(query_embedding, np.ndarray)
assert query_embedding.shape[0] == local_embeddings.shape[1]
print("Exercise 3 passed.")

In [ ]:
# Exercise 4
# Search documents for an OCR-related query.

results = ___

assert isinstance(results, pd.DataFrame)
assert len(results) == 3
assert "score" in results.columns
print("Exercise 4 passed.")

In [ ]:
# Exercise 5
# Normalize a vector.

raw_vector = np.array([3, 4])
normalized = ___

assert abs(np.linalg.norm(normalized) - 1.0) < 1e-9
print("Exercise 5 passed.")

In [ ]:
# Exercise 6
# Estimate storage for 100,000 vectors with 1536 dimensions.

storage = ___

assert storage["num_vectors"] == 100000
assert storage["dimensions"] == 1536
assert storage["mb"] > 0
print("Exercise 6 passed.")

In [ ]:
# Exercise 7
# Recommend similar documents to R001.

recommendations = ___

assert isinstance(recommendations, pd.DataFrame)
assert "R001" not in recommendations["doc_id"].tolist()
print("Exercise 7 passed.")

In [ ]:
# Exercise 8
# Evaluate retrieval on the test cases.

eval_results = ___

assert isinstance(eval_results, pd.DataFrame)
assert "hit" in eval_results.columns
print("Exercise 8 passed.")

## Solutions

<details>
<summary>Exercise 1 solution</summary>

```python
vector = np.array([0.1, 0.2, 0.3])
```

</details>

<details>
<summary>Exercise 2 solution</summary>

```python
score = cosine_similarity(a, b)
```

</details>

<details>
<summary>Exercise 3 solution</summary>

```python
query_embedding = embed_query_local("OCR invoice extraction")
```

</details>

<details>
<summary>Exercise 4 solution</summary>

```python
results = search_documents("OCR invoice fields", top_k=3)
```

</details>

<details>
<summary>Exercise 5 solution</summary>

```python
normalized = normalize_l2(raw_vector)
```

</details>

<details>
<summary>Exercise 6 solution</summary>

```python
storage = estimate_embedding_storage(100_000, 1536)
```

</details>

<details>
<summary>Exercise 7 solution</summary>

```python
recommendations = recommend_similar_documents("R001", local_embeddings, top_k=3)
```

</details>

<details>
<summary>Exercise 8 solution</summary>

```python
eval_results = evaluate_retrieval(test_cases, top_k=3)
```

</details>

## Cumulative review exercises

These mix topics from Days 16 to 25. Fill in `___` and run each cell.

In [ ]:
# Review 1: OpenAI API
# Fill the standard chat roles.

roles = ___

assert roles == ["system", "user", "assistant"]
print("Review 1 passed.")

In [ ]:
# Review 2: Ollama
# Fill the default local generate endpoint.

ollama_url = ___

assert ollama_url == "http://localhost:11434/api/generate"
print("Review 2 passed.")

In [ ]:
# Review 3: Prompt engineering
# Choose the prompting style that uses examples.

prompt_style = ___

assert prompt_style.lower() == "few-shot"
print("Review 3 passed.")

In [ ]:
# Review 4: Structured output
# Parse JSON text.

json_text = '{"campaign": "Demo", "clicks": 100}'
parsed = ___

assert parsed["clicks"] == 100
print("Review 4 passed.")

In [ ]:
# Review 5: Information extraction
# Calculate conversion rate.

record = {"clicks": 1000, "conversions": 75}
conversion_rate = ___

assert abs(conversion_rate - 0.075) < 1e-9
print("Review 5 passed.")

In [ ]:
# Review 6: Tesseract basics
# Choose the Tesseract language code for English.

english_lang_code = ___

assert english_lang_code == "eng"
print("Review 6 passed.")

In [ ]:
# Review 7: EasyOCR
# Create a language list for English and German.

easyocr_languages = ___

assert easyocr_languages == ["en", "de"] or easyocr_languages == ["de", "en"]
print("Review 7 passed.")

In [ ]:
# Review 8: OpenCV preprocessing
# Select the thresholding method useful for uneven lighting.

threshold_method = ___

assert threshold_method.lower() == "adaptive"
print("Review 8 passed.")

In [ ]:
# Review 9: OCR plus LLM pipeline
# Choose the final structured output format.

structured_format = ___

assert structured_format.upper() == "JSON"
print("Review 9 passed.")

In [ ]:
# Review 10: Document intelligence
# Create a document quality rule.

quality_rule = ___

assert "total" in quality_rule.lower() or "date" in quality_rule.lower() or "id" in quality_rule.lower()
print("Review 10 passed.")

## Cumulative review solutions

<details>
<summary>Show solutions</summary>

```python
# Review 1
roles = ["system", "user", "assistant"]

# Review 2
ollama_url = "http://localhost:11434/api/generate"

# Review 3
prompt_style = "few-shot"

# Review 4
parsed = json.loads(json_text)

# Review 5
conversion_rate = record["conversions"] / record["clicks"]

# Review 6
english_lang_code = "eng"

# Review 7
easyocr_languages = ["en", "de"]

# Review 8
threshold_method = "adaptive"

# Review 9
structured_format = "JSON"

# Review 10
quality_rule = "Invoice total must be present and non-negative."
```

</details>

In [ ]:
cheat_sheet = '''
DAY 26 CHEAT SHEET: EMBEDDINGS DEEP DIVE

Core idea:
- Embeddings turn text into numeric vectors.
- Similar meanings should have similar vectors.
- Semantic search compares query vectors with document vectors.

Cosine similarity:
- Measures similarity of vector direction.
- Range is usually from -1 to 1.
- Higher means more similar.
- If vectors are normalized, dot product gives the same ranking.

Local practice:
- TF-IDF can teach vector search locally.
- Hash embeddings are useful as a tiny fallback.
- sentence-transformers can create local neural embeddings.

OpenAI embeddings:
- Use client.embeddings.create(...)
- Common model examples: text-embedding-3-small and text-embedding-3-large
- Batch texts when possible.
- Store embeddings for reuse.

Production tips:
- Use the same model for documents and queries.
- Normalize vectors when your search method needs it.
- Estimate storage before indexing.
- Evaluate retrieval with test queries.
- Bad chunks lead to bad retrieval.
'''

print(cheat_sheet)

## Next up: Day 27 — ChunkingStrategies

You will learn fixed-size chunking, recursive character splitting, semantic chunking, and overlap.